# Automated Score Digitizer -- CNN Training Notebook

This notebook trains a Convolutional Neural Network (CNN) on the MNIST
handwritten digits dataset using PyTorch. The trained weights are saved
to `cnn_model.pth` for use in the companion Streamlit application.

---

**Dataset:** MNIST (28x28 grayscale images, 10 classes: digits 0-9)  
**Framework:** PyTorch  
**Architecture:** 2 convolutional blocks + 3 fully-connected layers  

## 1. Imports and Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# ---- Hyperparameters ----
BATCH_SIZE = 128        # Number of samples per mini-batch
LEARNING_RATE = 1e-3    # Adam optimizer learning rate
NUM_EPOCHS = 10         # Total training epochs
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

## 2. Dataset Preparation

We apply the standard MNIST normalisation (mean=0.1307, std=0.3081)
which centres the pixel distribution around zero. This accelerates
convergence and stabilises gradient flow through the network.

In [ ]:
# Define the preprocessing pipeline:
# 1. Convert PIL image to tensor (scales pixel values from [0,255] to [0,1])
# 2. Normalise using MNIST channel statistics
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])

# Download and load the MNIST training and test splits
train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# Wrap datasets in DataLoaders for batched iteration
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")

## 3. Visualise Sample Data

In [ ]:
# Display a grid of 16 training samples to verify the data pipeline
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    # image shape is (1, 28, 28); squeeze to (28, 28) for display
    ax.imshow(image.squeeze().numpy(), cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.suptitle("MNIST Training Samples")
plt.tight_layout()
plt.show()

## 4. Model Architecture

The CNN follows a classic pattern of alternating convolution and pooling
layers to extract spatial features, followed by dense layers for
classification.

**Feature Extraction (Convolutional Blocks):**
- Block 1: Conv2d(1->32) -> BatchNorm -> ReLU -> MaxPool  
- Block 2: Conv2d(32->64) -> BatchNorm -> ReLU -> MaxPool  

**Classifier (Fully-Connected Layers):**
- FC(3136 -> 128) -> ReLU -> Dropout(0.25)  
- FC(128 -> 64) -> ReLU  
- FC(64 -> 10) -- raw logits for each digit class  

In [ ]:
class DigitCNN(nn.Module):
    """
    Convolutional Neural Network for classifying 28x28 grayscale images
    of handwritten digits (0-9).

    The architecture consists of two convolutional blocks for hierarchical
    feature extraction followed by a three-layer classifier.
    """

    def __init__(self):
        super(DigitCNN, self).__init__()

        # ---------------------------------------------------------------
        # CONVOLUTIONAL FEATURE EXTRACTOR
        # ---------------------------------------------------------------
        self.conv_block = nn.Sequential(
            # --- Block 1 ---
            # Conv2d: Applies 32 learnable 3x3 filters over the input.
            #   - in_channels=1  : single grayscale channel
            #   - out_channels=32: produces 32 feature maps
            #   - padding=1      : 'same' padding preserves spatial dims
            # Output shape: (batch, 32, 28, 28)
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),

            # BatchNorm2d: Normalises activations across the batch for each
            # of the 32 channels.  This stabilises training and allows
            # higher learning rates.
            nn.BatchNorm2d(32),

            # ReLU: Rectified Linear Unit activation -- introduces
            # non-linearity by zeroing out negative values.
            # inplace=True saves memory by modifying the tensor directly.
            nn.ReLU(inplace=True),

            # MaxPool2d: Downsamples spatial dimensions by 2x, retaining
            # the maximum activation in each 2x2 window.
            # Output shape: (batch, 32, 14, 14)
            nn.MaxPool2d(kernel_size=2, stride=2),

            # --- Block 2 ---
            # Doubles the channel depth while halving spatial resolution.
            # Output after conv: (batch, 64, 14, 14)
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            # Output after pool: (batch, 64, 7, 7)
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # ---------------------------------------------------------------
        # FULLY-CONNECTED CLASSIFIER
        # ---------------------------------------------------------------
        self.fc_block = nn.Sequential(
            # Flatten: reshapes (batch, 64, 7, 7) -> (batch, 3136)
            nn.Flatten(),

            # FC layer 1: reduces 3136 features to 128
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),

            # Dropout: randomly zeroes 25% of activations during training
            # to prevent co-adaptation and reduce overfitting.
            nn.Dropout(p=0.25),

            # FC layer 2: further compression to 64 features
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),

            # FC layer 3 (output): maps to 10 raw logits, one per class.
            # No softmax here -- CrossEntropyLoss applies it internally.
            nn.Linear(64, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Args:
            x: Input tensor of shape (batch, 1, 28, 28).

        Returns:
            Logits tensor of shape (batch, 10).
        """
        x = self.conv_block(x)   # extract spatial features
        x = self.fc_block(x)     # classify
        return x


# Instantiate the model and move it to the compute device
model = DigitCNN().to(DEVICE)

# Print architecture summary
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {total_params:,}")

## 5. Loss Function and Optimizer

- **CrossEntropyLoss**: Combines LogSoftmax and NLLLoss in a single step.
  It is the standard loss for multi-class classification.
- **Adam**: Adaptive Moment Estimation optimizer -- maintains per-parameter
  learning rates using first and second moment estimates of the gradients.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

## 6. Training Loop

In [ ]:
# Storage for plotting learning curves
train_losses = []
train_accuracies = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()  # enable dropout and batch-norm in training mode

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        # Move data to the compute device (CPU or GPU)
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # --- Forward pass ---
        outputs = model(images)           # raw logits (batch, 10)
        loss = criterion(outputs, labels) # cross-entropy loss

        # --- Backward pass ---
        optimizer.zero_grad()  # clear accumulated gradients
        loss.backward()        # compute gradients via backpropagation
        optimizer.step()       # update weights using Adam

        # --- Accumulate statistics ---
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    # Compute epoch-level metrics
    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)

    print(
        f"Epoch [{epoch:2d}/{NUM_EPOCHS}]  "
        f"Loss: {epoch_loss:.4f}  "
        f"Accuracy: {epoch_acc:.2f}%"
    )

print("\nTraining complete.")

## 7. Learning Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, NUM_EPOCHS + 1), train_losses, marker="o", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, NUM_EPOCHS + 1), train_accuracies, marker="o", linewidth=2, color="green")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Training Accuracy")
ax2.grid(True, alpha=0.3)

plt.suptitle("Training Learning Curves")
plt.tight_layout()
plt.show()

## 8. Evaluation on Test Set

In [ ]:
model.eval()  # disable dropout and set batch-norm to inference mode

test_correct = 0
test_total = 0

with torch.no_grad():  # disable gradient computation for efficiency
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs, dim=1)
        test_correct += (predicted == labels).sum().item()
        test_total += labels.size(0)

test_accuracy = 100.0 * test_correct / test_total
print(f"Test Accuracy: {test_accuracy:.2f}% ({test_correct}/{test_total})")

## 9. Visualise Predictions on Test Samples

In [ ]:
# Grab one batch from the test loader
sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images.to(DEVICE)

with torch.no_grad():
    sample_outputs = model(sample_images)
    _, sample_preds = torch.max(sample_outputs, dim=1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = sample_images[i].cpu().squeeze().numpy()
    true_label = sample_labels[i].item()
    pred_label = sample_preds[i].item()
    ax.imshow(img, cmap="gray")
    colour = "green" if true_label == pred_label else "red"
    ax.set_title(f"T:{true_label} P:{pred_label}", color=colour)
    ax.axis("off")

plt.suptitle("Test Predictions (green = correct, red = incorrect)")
plt.tight_layout()
plt.show()

## 10. Save Trained Model Weights

In [ ]:
# Save only the state_dict (model weights), not the entire model object.
# This is the PyTorch-recommended serialisation approach.
MODEL_PATH = "cnn_model.pth"
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model weights saved to: {MODEL_PATH}")